In [1]:
# 03_model_training.ipynb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import os

In [2]:
#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
# 載入處理後數據
#processed_path = "/content/data/processed/elliptic_processed.pt" # google colab
processed_path = "../data/processed/elliptic_processed.pt" # local

data = torch.load(processed_path)
x = data['x']
edge_index = data['edge_index']
y = data['y']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = x.to(device)
edge_index = edge_index.to(device)
y = y.to(device)

/var/folders/6p/f3bb7wps1p9crqh3cv0vjjfc0000gn/T/ipykernel_37169/938148345.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(processed_path)


In [4]:
# 拆分 train/test（只用有標記的）
labeled_mask = y != -1
labeled_indices = torch.where(labeled_mask)[0].cpu().numpy()

train_idx, test_idx = train_test_split(
    labeled_indices,
    test_size=0.2,
    stratify=y[labeled_mask].cpu().numpy(),
    random_state=42
)

train_idx = torch.tensor(train_idx, device=device)
test_idx = torch.tensor(test_idx, device=device)

print(f"訓練樣本: {len(train_idx)}, 測試樣本: {len(test_idx)}")

訓練樣本: 37251, 測試樣本: 9313


In [5]:
# 定義 GraphSAGE 模型
class GraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden=128, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(in_feats, hidden))
        for _ in range(1, num_layers):
            self.layers.append(nn.Linear(hidden * 2, hidden))
        self.classifier = nn.Linear(hidden, 2)  # 正常/異常

    def forward(self, x, edge_index, idx=None):
        h = x
        for layer in self.layers:
            row, col = edge_index
            num_neigh = torch.bincount(row, minlength=h.size(0)).float()
            aggr = torch.zeros_like(h)
            aggr.index_add_(0, row, h[col])
            aggr[row.unique()] /= (num_neigh[row.unique()] + 1e-10).unsqueeze(1)
            h = torch.cat([h, aggr], dim=1)
            h = F.relu(layer(h))
        logits = self.classifier(h)
        if idx is not None:
            return logits[idx]
        return logits

model = GraphSAGE(in_feats=x.shape[1]).to(device)
print(model)

GraphSAGE(
  (layers): ModuleList(
    (0): Linear(in_features=167, out_features=128, bias=True)
    (1): Linear(in_features=256, out_features=128, bias=True)
  )
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [6]:
# 訓練
optimizer = Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    logits = model(x, edge_index, train_idx)
    loss = loss_fn(logits, y[train_idx])
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (203769x334 and 167x128)

In [ ]:
# 評估
model.eval()
with torch.no_grad():
    logits = model(x, edge_index)
    probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()

test_probs = probs[test_idx.cpu().numpy()]
test_labels = y[test_idx].cpu().numpy()

auc = roc_auc_score(test_labels, test_probs)
auprc = average_precision_score(test_labels, test_probs)

print(f"\n測試集結果：")
print(f"AUC-ROC: {auc:.4f}")
print(f"AUPRC   : {auprc:.4f}")